In [1]:
!pip install -q -U google-genai pandas python-docx requests xmltodict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.

In [28]:
import json
from google import genai
from google.genai import types
from google.genai.errors import ClientError, ServerError, APIError # Import specific error types
from google.colab import userdata
import time # For retries

# 1. Khởi tạo kết nối Gemini
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') # Lấy API key từ Colab Secrets
client = genai.Client(api_key=GEMINI_API_KEY)

# Cho phép người dùng chọn mô hình Gemini ban đầu
preferred_model = input("📌 Nhập tên mô hình Gemini bạn muốn ưu tiên sử dụng (ví dụ: gemini-flash-latest, gemini-pro-latest). Nhấn Enter để dùng mặc định: ").strip()

# Danh sách các mô hình Gemini có sẵn để thử, theo thứ tự ưu tiên
# Các mô hình được lấy từ output của client.models.list() và thường có trong gói miễn phí
AVAILABLE_GEMINI_MODELS = [
    'gemini-flash-latest',       # Phiên bản flash mới nhất
    'gemini-pro-latest',         # Phiên bản pro mới nhất
    'gemini-3.5-flash',          # Một phiên bản flash cụ thể
    'gemini-3.5-flash-lite',     # Một phiên bản flash nhẹ cụ thể
    'gemini-1.0-pro'             # Phiên bản 1.0 pro ổn định
]

# Nếu người dùng nhập mô hình, thêm nó vào đầu danh sách nếu chưa có
if preferred_model and preferred_model not in AVAILABLE_GEMINI_MODELS:
    AVAILABLE_GEMINI_MODELS.insert(0, preferred_model)
elif preferred_model and preferred_model in AVAILABLE_GEMINI_MODELS:
    # Nếu mô hình ưa thích đã có trong danh sách, di chuyển nó lên đầu
    AVAILABLE_GEMINI_MODELS.remove(preferred_model)
    AVAILABLE_GEMINI_MODELS.insert(0, preferred_model)


current_model_idx = 0
MAX_RETRIES_PER_MODEL = 2 # Số lần thử lại với cùng một mô hình trước khi chuyển sang mô hình khác

def call_gemini_with_fallback(prompt_contents, prompt_config=None):
    global current_model_idx
    model_retries_count = 0

    while current_model_idx < len(AVAILABLE_GEMINI_MODELS):
        model_name = AVAILABLE_GEMINI_MODELS[current_model_idx]
        print(f"✨ Đang thử gọi Gemini model: {model_name} (Lần thử {model_retries_count + 1}/{MAX_RETRIES_PER_MODEL + 1} trên mô hình này)")
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt_contents,
                config=prompt_config
            )
            # Nếu thành công, đặt lại số lần thử cho mô hình này cho các cuộc gọi sau
            model_retries_count = 0
            return response
        except APIError as e: # Catch APIError which is the base for ClientError and ServerError
            if hasattr(e, 'response') and hasattr(e.response, 'status_code'):
                status_code = e.response.status_code
            else:
                print(f"⚠️ Lỗi APIError không có status_code với {model_name}: {e}. Chuyển sang mô hình tiếp theo.")
                current_model_idx += 1
                continue

            if status_code == 429: # Resource Exhausted (Quota limit)
                print(f"⚠️ Lỗi 429 (Resource Exhausted) với {model_name}. ")
                model_retries_count += 1
                if model_retries_count <= MAX_RETRIES_PER_MODEL:
                    print(f"Thử lại mô hình {model_name} sau {2 ** model_retries_count} giây...")
                    time.sleep(2 ** model_retries_count) # Exponential backoff
                    continue
                else:
                    print(f"Hết số lần thử cho mô hình {model_name}. Chuyển sang mô hình tiếp theo.")
                    model_retries_count = 0 # Đặt lại số lần thử cho mô hình tiếp theo
                    current_model_idx += 1
            elif status_code == 503: # Service Unavailable (Model overload)
                print(f"⚠️ Lỗi 503 (Service Unavailable) với {model_name}. ")
                model_retries_count += 1
                if model_retries_count <= MAX_RETRIES_PER_MODEL:
                    print(f"Thử lại mô hình {model_name} sau {2 ** model_retries_count} giây...")
                    time.sleep(2 ** model_retries_count)
                    continue
                else:
                    print(f"Hết số lần thử cho mô hình {model_name}. Chuyển sang mô hình tiếp theo.")
                    model_retries_count = 0
                    current_model_idx += 1
            elif status_code == 404: # Not Found (Model deprecated/unavailable)
                print(f"❌ Mô hình {model_name} không tìm thấy hoặc không khả dụng. Chuyển sang mô hình tiếp theo.")
                current_model_idx += 1
                model_retries_count = 0 # Đặt lại số lần thử cho mô hình tiếp theo
                continue
            else:
                print(f"❌ Lỗi Gemini API khác ({status_code}) với {model_name}: {e}. Không thể tiếp tục.")
                raise e # Re-raise other unexpected errors
        except Exception as e:
            print(f"❌ Một lỗi không mong muốn xảy ra: {e}. Không thể tiếp tục.")
            raise e

    print("🚫 Đã thử tất cả các mô hình Gemini có sẵn và đều gặp lỗi. Vui lòng kiểm tra lại quota hoặc thử lại sau.")
    raise Exception("Không thể gọi Gemini API với bất kỳ mô hình nào.")


# 2. Nhập tên đề tài
RAW_TOPIC = input("📌 Nhập tên đề tài hoặc câu hỏi nghiên cứu của bạn: ").strip()
print("\n⏳ Đang phân tích đề tài, trích xuất PICO và tạo search query chuẩn...")

# 3. Yêu cầu Gemini phân tích phương pháp luận
prompt_protocol = f"""
Bạn là chuyên gia phương pháp luận tổng quan hệ thống (Systematic Review).
Hãy phân tích đề tài: "{RAW_TOPIC}"

Nhiệm vụ:
1. Bóc tách PICO (Population, Intervention, Comparison, Outcome).
2. Xây dựng Tiêu chuẩn chọn (Inclusion) và Tiêu chuẩn loại (Exclusion).
3. Viết Search Query chuẩn cho PubMed (dùng toán tử AND, OR, ngoặc đơn [tiab]).
4. Đề xuất khoảng năm quét (mặc định 2020:2026).

Trả về định dạng JSON thuần túy:
{{
  "P": "Đối tượng nghiên cứu",
  "I": "Can thiệp / Phơi nhiễm",
  "C": "So sánh / Đối chứng",
  "O": "Kết quả đầu ra chính",
  "inclusion_criteria": "Danh sách gạch đầu dòng tiêu chí chọn",
  "exclusion_criteria": "Danh sách gạch đầu dòng tiêu chí loại",
  "search_query": "Chuỗi tìm kiếm tối ưu cho PubMed",
  "suggested_years": "2020:2026"
}}
"""

protocol_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.2
)

response = call_gemini_with_fallback(prompt_protocol, protocol_config)

protocol = json.loads(response.text)
max_results = 20  # Mặc định 20 bài cho bản thử nghiệm

# 4. Hiển thị đề xuất lên màn hình (Checkpoint 0)
print("\n" + "="*65)
print("🎯 BẢNG ĐỀ XUẤT PHƯƠNG PHÁP LUẬN TỰ ĐỘNG (CHECKPOINT 0):")
print("="*65)
print(f"• P (Population):   {protocol['P']}")
print(f"• I (Intervention): {protocol['I']}")
print(f"• C (Comparison):   {protocol['C']}")
print(f"• O (Outcome):      {protocol['O']}")
print(f"\n[+] Tiêu chuẩn chọn:\n{protocol['inclusion_criteria']}")
print(f"\n[-] Tiêu chuẩn loại:\n{protocol['exclusion_criteria']}")
print(f"\n🔍 Search Query PubMed:\n{protocol['search_query']}")
print(f"📅 Khoảng năm: {protocol['suggested_years']}")
print(f"📑 Số lượng bài báo tối đa cần quét: {max_results} bài")
print("="*65)

# 5. Phê duyệt hoặc chỉnh sửa
action = input("\n👉 Duyệt cấu hình này? (Nhấn Enter hoặc gõ 'yes' để duyệt / Gõ 'edit' để sửa): ").strip().lower()

if action == 'edit':
    print("\n--- CHỈNH SỬA THÔNG SỐ (Nhấn Enter để giữ nguyên giá trị cũ) ---")
    custom_query = input(f"Search Query mới [{protocol['search_query']}]: ").strip()
    if custom_query:
        protocol['search_query'] = custom_query

    custom_years = input(f"Khoảng năm mới [{protocol['suggested_years']}]: ").strip()
    if custom_years:
        protocol['suggested_years'] = custom_years

    custom_max = input(f"Số lượng bài tối đa cần quét [{max_results}]: ").strip()
    if custom_max and custom_max.isdigit():
        max_results = int(custom_max)

    print(f"\n✅ Đã cập nhật: Quét tối đa {max_results} bài với thông số tùy chỉnh.")
elif action == 'yes':
    print(f"\n✅ Đã phê duyệt cấu hình. Tiến hành quét PubMed...")
else:
    # Default to 'yes' if user just presses Enter
    print(f"\n✅ Đã phê duyệt cấu hình. Tiến hành quét PubMed...")

# Gán biến hệ thống
TOPIC_QUERY = f"({protocol['search_query']}) AND ({protocol['suggested_years']}[dp])"
INCLUSION_CRITERIA = protocol['inclusion_criteria']
EXCLUSION_CRITERIA = protocol['exclusion_criteria']
MAX_RESULTS = max_results

📌 Nhập tên mô hình Gemini bạn muốn ưu tiên sử dụng (ví dụ: gemini-flash-latest, gemini-pro-latest). Nhấn Enter để dùng mặc định: 
📌 Nhập tên đề tài hoặc câu hỏi nghiên cứu của bạn: AI testing

⏳ Đang phân tích đề tài, trích xuất PICO và tạo search query chuẩn...
✨ Đang thử gọi Gemini model: gemini-flash-latest (Lần thử 1/3 trên mô hình này)
⚠️ Lỗi 503 (Service Unavailable) với gemini-flash-latest. 
Thử lại mô hình gemini-flash-latest sau 2 giây...
✨ Đang thử gọi Gemini model: gemini-flash-latest (Lần thử 2/3 trên mô hình này)
⚠️ Lỗi 503 (Service Unavailable) với gemini-flash-latest. 
Thử lại mô hình gemini-flash-latest sau 4 giây...
✨ Đang thử gọi Gemini model: gemini-flash-latest (Lần thử 3/3 trên mô hình này)
⚠️ Lỗi 503 (Service Unavailable) với gemini-flash-latest. 
Hết số lần thử cho mô hình gemini-flash-latest. Chuyển sang mô hình tiếp theo.
✨ Đang thử gọi Gemini model: gemini-pro-latest (Lần thử 1/3 trên mô hình này)
⚠️ Lỗi 429 (Resource Exhausted) với gemini-pro-latest. 
Thử lại

In [17]:
AVAILABLE_GEMINI_MODELS = [
    'gemini-1.0-pro',
    'gemini-1.5-flash',
    'gemini-1.5-pro' # Lưu ý: Mô hình 1.5-pro có thể có chi phí cao hơn và hạn chế về quota
]
current_model_idx = 0
print(f"✅ Đã cập nhật danh sách mô hình khả dụng thành: {AVAILABLE_GEMINI_MODELS}")

✅ Đã cập nhật danh sách mô hình khả dụng thành: ['gemini-1.0-pro', 'gemini-1.5-flash', 'gemini-1.5-pro']


In [26]:
# Kiểm tra các mô hình Gemini có sẵn và các thuộc tính của chúng
print("Đang lấy danh sách các mô hình Gemini khả dụng...")
for m in client.models.list():
    # Chỉ in tên mô hình để người dùng tự xem xét, không lọc theo supported_generation_methods nữa.
    print(m.name)


Đang lấy danh sách các mô hình Gemini khả dụng...
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robo

In [29]:
import pandas as pd
import requests
import xmltodict

def search_pubmed(query, max_results=20):
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    # Bước 1: Tìm danh sách ID bài báo (PMID)
    search_url = f"{base_url}esearch.fcgi?db=pubmed&term={query}&retmax={max_results}&retmode=json"
    res = requests.get(search_url).json()
    id_list = res.get('esearchresult', {}).get('idlist', [])

    if not id_list:
        print("⚠️ Không tìm thấy bài báo nào phù hợp với Search Query.")
        return pd.DataFrame()

    # Bước 2: Tải thông tin chi tiết (XML)
    fetch_url = f"{base_url}efetch.fcgi?db=pubmed&id={','.join(id_list)}&retmode=xml"
    xml_data = requests.get(fetch_url).content
    data_dict = xmltodict.parse(xml_data)

    articles = []
    pubmed_articles = data_dict['PubmedArticleSet'].get('PubmedArticle', [])
    if isinstance(pubmed_articles, dict):
        pubmed_articles = [pubmed_articles]

    for art in pubmed_articles:
        medline = art['MedlineCitation']
        article_info = medline['Article']
        title = article_info.get('ArticleTitle', '')

        abstract_raw = article_info.get('Abstract', {}).get('AbstractText', '')
        if isinstance(abstract_raw, list):
            abstract = " ".join([item.get('#text', str(item)) if isinstance(item, dict) else str(item) for item in abstract_raw])
        elif isinstance(abstract_raw, dict):
            abstract = abstract_raw.get('#text', '')
        else:
            abstract = str(abstract_raw)

        pmid = medline.get('PMID', {}).get('#text', '')
        articles.append({
            "PMID": pmid,
            "Title": title,
            "Abstract": abstract
        })

    df = pd.DataFrame(articles)
    df.to_csv("records_raw.csv", index=False)
    print(f"✅ Đã tải thành công {len(df)} bài báo từ PubMed vào file 'records_raw.csv'.")
    return df

df_records = search_pubmed(TOPIC_QUERY, max_results=MAX_RESULTS)
df_records.head(3)

✅ Đã tải thành công 20 bài báo từ PubMed vào file 'records_raw.csv'.


,PMID,Title,Abstract
0,42608355,Intermuscular Adipose Tissue as a Multimodalit...,Intermuscular adipose tissue (IMAT) refers to ...
1,42608332,Integrative Proteomics-Metabolomics Profiling ...,Gestational diabetes mellitus (GDM) is a commo...
2,42608308,[Construction of a genotype-phenotype database...,To establish a genotype-phenotype database and...


In [33]:
import re
screening_results = []
print("⏳ Đang tiến hành sàng lọc Title & Abstract qua Gemini...")

for idx, row in df_records.iterrows():
    prompt_screen = f"""
    Bạn là chuyên gia thẩm định y văn. Đối chiếu bài báo sau với tiêu chuẩn chọn/loại:

    TIÊU CHUẨN CHỌN:
    {INCLUSION_CRITERIA}

    TIÊU CHUẨN LOẠI:
    {EXCLUSION_CRITERIA}

    BÀI BÁO:
    - Tiêu đề: {row['Title']}
    - Tóm tắt: {row['Abstract']}

    Trả về đúng định dạng JSON:
    {{
      "decision": "INCLUDE" hoặc "EXCLUDE" hoặc "UNCERTAIN",
      "reason": "Lý do ngắn gọn trong 1 câu"
    }}
    """

    screen_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1
    )

    # Sử dụng hàm dự phòng để gọi Gemini API
    response_screen = call_gemini_with_fallback(prompt_screen, screen_config)

    # Xử lý phản hồi để đảm bảo chỉ có JSON hợp lệ
    raw_text = response_screen.text.strip()
    res_data = None

    try:
        res_data = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"⚠️ Cảnh báo: Lỗi phân tích JSON trực tiếp: {e}. Đang thử khắc phục...")

        # Heuristic 1: Tìm JSON block bằng regex linh hoạt hơn
        # Sử dụng non-greedy match để tránh khớp quá nhiều nếu có nhiều JSON hoặc văn bản không liên quan
        json_match = re.search(r'\{.*?\}', raw_text, re.DOTALL)

        if json_match:
            json_str = json_match.group(0)
            try:
                res_data = json.loads(json_str)
            except json.JSONDecodeError as e_regex:
                print(f"⚠️ Cảnh báo: Lỗi phân tích JSON sau khi trích xuất bằng regex: {e_regex}. Đang thử khắc phục thêm...")
                # Heuristic 2: LLM đôi khi thêm dấu } thừa
                if json_str.endswith('}}'):
                    try:
                        res_data = json.loads(json_str[:-1]) # Thử bỏ ký tự '}' cuối cùng
                    except json.JSONDecodeError:
                        pass # Nếu vẫn lỗi, không làm gì và để mặc định

                # Heuristic 3: LLM đôi khi thiếu dấu } (dựa vào raw_text)
                if res_data is None and raw_text.startswith('{') and not raw_text.endswith('}'):
                    try:
                        # Cố gắng hoàn thành JSON bằng cách thêm dấu đóng ngoặc
                        temp_json_str = raw_text + '}'
                        res_data = json.loads(temp_json_str)
                        print(f"✅ Đã khắc phục JSON bị thiếu dấu đóng ngoặc.")
                    except json.JSONDecodeError:
                        pass # Nếu vẫn lỗi, không làm gì

        if res_data is None: # Nếu sau các heuristic vẫn không có JSON hợp lệ
            print(f"⚠️ Cảnh báo: Không thể phân tích JSON hợp lệ từ phản hồi:
{raw_text}\nSử dụng giá trị mặc định.")

    if res_data is None: # Nếu tất cả các cách đều thất bại (hoặc không tìm thấy match ban đầu)
        res_data = {"decision": "UNCERTAIN", "reason": "Phản hồi không phải JSON hợp lệ"}

    screening_results.append({
        "PMID": row['PMID'],
        "Title": row['Title'],
        "Abstract": row['Abstract'],
        "AI_Decision": res_data['decision'],
        "AI_Reason": res_data['reason'],
        "Human_Decision": res_data['decision']  # Cột dành cho người dùng rà soát
    })

df_screening = pd.DataFrame(screening_results)
df_screening.to_csv("checkpoint1_screening.csv", index=False)

print("\n🎯 KẾT QUẢ SÀNG LỌC CHECKPOINT 1:")
print(f"• Tổng số bài: {len(df_screening)}")
print(f"• INCLUDE:   {len(df_screening[df_screening['AI_Decision']=='INCLUDE'])}")
print(f"• EXCLUDE:   {len(df_screening[df_screening['AI_Decision']=='EXCLUDE'])}")
print(f"• UNCERTAIN: {len(df_screening[df_screening['AI_Decision']=='UNCERTAIN'])}")
print("\n👉 Bảng kết quả đã được lưu tại 'checkpoint1_screening.csv'")
df_screening[['PMID', 'AI_Decision', 'AI_Reason']].head(5)

⏳ Đang tiến hành sàng lọc Title & Abstract qua Gemini...
✨ Đang thử gọi Gemini model: gemini-3.5-flash (Lần thử 1/3 trên mô hình này)
✨ Đang thử gọi Gemini model: gemini-3.5-flash (Lần thử 1/3 trên mô hình này)
✨ Đang thử gọi Gemini model: gemini-3.5-flash (Lần thử 1/3 trên mô hình này)
⚠️ Cảnh báo: Không tìm thấy đối tượng JSON trong phản hồi từ Gemini: {
  "decision": "EXCLUDE",
  "reason": "Nghiên cứu này tập trung vào việc xây dựng và huấn luyện mô hình học máy để dự đoán kiểu hình lâm sàng thay vì phát triển phương pháp luận, khung hoặc công cụ kiểm thử và đánh giá chất lượng hệ thống AI/ML."
⚠️ Cảnh báo: Phản hồi từ Gemini không phải là JSON hợp lệ. Sử dụng giá trị mặc định.
✨ Đang thử gọi Gemini model: gemini-3.5-flash (Lần thử 1/3 trên mô hình này)
⚠️ Cảnh báo: Không tìm thấy đối tượng JSON trong phản hồi từ Gemini: {
  "decision": "INCLUDE",
  "reason": "Nghiên cứu thực hiện đánh giá thuật toán (algorithmic audit) đối với ChatGPT-5.1 trong việc tạo kế hoạch chăm sóc sau điều t

,PMID,AI_Decision,AI_Reason
0,42608355,EXCLUDE,Bài báo là một bài tổng quan (review) về dấu ấ...
1,42608332,EXCLUDE,Nghiên cứu tập trung vào việc ứng dụng học máy...
2,42608308,UNCERTAIN,Phản hồi không phải JSON hợp lệ
3,42608279,UNCERTAIN,Phản hồi không phải JSON hợp lệ
4,42608170,EXCLUDE,Bài báo là một bài tổng hợp mang tính quan điể...


In [34]:
from docx import Document
from docx.shared import Pt, Inches

# 1. Lọc danh sách bài được chọn
included_papers = df_screening[df_screening['Human_Decision'] == 'INCLUDE']

if len(included_papers) == 0:
    print("⚠️ Không có bài báo nào ở trạng thái INCLUDE. Vui lòng kiểm tra lại Checkpoint 1.")
else:
    print(f"⏳ Đang tiến hành phân tích tổng hợp từ {len(included_papers)} bài báo...")

    synthesis_input = ""
    for idx, row in included_papers.iterrows():
        synthesis_input += f"\n--- BÀI BÁO [PMID: {row['PMID']}] ---\nTiêu đề: {row['Title']}\nTóm tắt: {row['Abstract']}\n"

    prompt_write = f"""
    Bạn là chuyên gia viết tổng quan tài liệu y học. Dựa trên các nghiên cứu sau đây:
    {synthesis_input}

    Hãy viết một bản Tổng quan y văn hoàn chỉnh bằng tiếng Việt theo cấu trúc sau:
    1. Đặt vấn đề & Mục tiêu tổng quan
    2. Phương pháp luận tóm tắt (Khung PICO và tiêu chí lựa chọn)
    3. Tổng hợp bằng chứng theo chủ đề (Đối chiếu điểm tương đồng, mâu thuẫn giữa các bài báo; bắt buộc trích dẫn [PMID: ...])
    4. Bàn luận & Khoảng trống nghiên cứu (Research Gaps)
    5. Kết luận
    6. Tóm tắt (Abstract) viết ở cuối dựa trên toàn bộ nội dung đã viết.
    """

    write_config = types.GenerateContentConfig(temperature=0.3)

    response_write = call_gemini_with_fallback(prompt_write, write_config)

    # 2. Định dạng và xuất file Word
    doc = Document()
    title_heading = doc.add_heading(f"TỔNG QUAN TÀI LIỆU: {RAW_TOPIC.upper()}", 0)
    title_heading.alignment = 1

    # Metadata phần mở đầu
    doc.add_paragraph(f"Chuỗi tìm kiếm: {TOPIC_QUERY}")
    doc.add_paragraph(f"Số lượng nghiên cứu đưa vào tổng quan: {len(included_papers)} bài\n")

    # Nội dung chính
    for p in response_write.text.split('\n\n'):
        if p.strip():
            doc.add_paragraph(p.strip())

    doc.save("Tong_quan_tai_lieu.docx")
    print("🎉 HOÀN TẤT QUY TRÌNH!")
    print("📁 Tải file 'Tong_quan_tai_lieu.docx' và 'checkpoint1_screening.csv' tại thanh công cụ Files (biểu tượng thư mục) bên trái màn hình Colab.")

⏳ Đang tiến hành phân tích tổng hợp từ 2 bài báo...
✨ Đang thử gọi Gemini model: gemini-3.5-flash-lite (Lần thử 1/3 trên mô hình này)
🎉 HOÀN TẤT QUY TRÌNH!
📁 Tải file 'Tong_quan_tai_lieu.docx' và 'checkpoint1_screening.csv' tại thanh công cụ Files (biểu tượng thư mục) bên trái màn hình Colab.
